# ERK-KTR Full FOV Stimulation Pipeline
This notebook showcases how to use the ERK-KTR full FOV stimulation pipeline. The pipeline is designed to simulate the full field of view (FOV) stimulation of a cells with the ERK-KTR biosensor. As it is a demo experiment, the pipeline runs on the demo hardware provided by MicroManager. 

### Import required libraries

In [1]:
import os
import time

os.environ["QT_LOGGING_RULES"] = (
    "*.debug=false; *.warning=false"  # Fix to suppress PyQT warnings from napari-micromanager when running in a Jupyter notebook
)
os.environ["MICROMANAGER_PATH"] = "C:\\Program Files\\Micro-Manager-2.0"

from rtm_pymmcore.data_structures import Fov, Channel, StimTreatment
from pprint import pprint
import pandas as pd
import numpy as np
import dataclasses
import random
import pymmcore_plus

mmc = pymmcore_plus.CMMCorePlus()

### Experimental Settings

In [ ]:
## Configuration options
N_FRAMES = 120
FIRST_FRAME_STIMULATION = 10

SLEEP_BEFORE_EXPERIMENT_START_in_H = 0
USE_AUTOFOCUS_EVENT = False
USE_ONLY_PFS = False

TIME_BETWEEN_TIMESTEPS = 60  # time in seconds between frames
TIME_PER_FOV = 6  # time in seconds per fov


## Storage path for the experiment
base_path = (
    "\\\\izbkingston.unibe.ch\\imaging.data\\mic01-imaging\\Cedric\\experimental_data\\"
)
experiment_name = "TestConfluency"
path = os.path.join(base_path, experiment_name)

# Define Channels for which Images are taken. If no power is defined, the default power of the device will be used,
# for example, see the second channel "Cy5" below. The default power is set in the GUI
channels = []
channels.append(Channel(name="miRFP", exposure=300))
channels.append(Channel(name="mScarlet3", exposure=100))

# Channel to check for the expression of the optogenetic marker, can be used if it the marker is in the same channel as the stimulation channel.
channel_optocheck = Channel(name="mCitrine", exposure=150)


# Condition mapping to FOVs. This is used to create a dataframe with the conditions and the FOVs.
condition = [
    "FGFR_high",
    "EGFR",
]  # Example of adding a condition to the dataframe. Stimulation will be repeated for each condition.
# condition = ["optoFGFR_high"] * 24 + ["optoFGFR"] * 24 # Example of adding multiple conditions to the dataframe. n repreats the amount of times the condition is repeated.

n_fovs_per_condition = 4  ## change this variable to the amount of fovs that you have per cell line. If only one cell line is set, this value will
# automatically set to total amount of fovs.

n_fovs_per_well = 4  ## change this variable to the amount of fovs that you have per well. Set to None if you are not working with wellplate.


# Stimulation parameters for optogenetics. The stimulation will be repeated for each condition.

stim_exposures_timesteps = [
    {
        "ramp_pattern_name": "ramp1",
        "stim_exposure_list": (10, 20, 30),
        "stim_timestep": (1, 2, 3),
    },
    {
        "ramp_pattern_name": "ramp2",
        "stim_exposure_list": range(100, 500, 50),
        "stim_timestep": (4, 5, 6, 7),
    },
]

if n_fovs_per_condition is not None and n_fovs_per_condition < len(
    stim_exposures_timesteps
):
    raise ValueError(
        "The number of fovs per condition is less than the number of stimulation patterns. Please increase the number of fovs per condition."
    )

for stim_exposure_timestep in stim_exposures_timesteps:
    if isinstance(stim_exposure_timestep["stim_timestep"], range):
        stim_exposure_timestep["stim_timestep"] = tuple(
            stim_exposure_timestep["stim_timestep"]
        )
    if isinstance(stim_exposure_timestep["stim_exposure_list"], range):
        stim_exposure_timestep["stim_exposure_list"] = tuple(
            stim_exposure_timestep["stim_exposure_list"]
        )

for stexptim in stim_exposures_timesteps:
    print("Pattern Name: ", stexptim["ramp_pattern_name"])

    for stim_exp, stim_timestep in zip(
        stexptim["stim_exposure_list"], stexptim["stim_timestep"]
    ):
        print(f"{stim_exp} at {stim_timestep}")
    print("")

    # add general stimulation parameters
    stexptim["stim_power"] = 11
    stexptim["stim_channel_name"] = "CyanStim"
    stexptim["stim_channel_group"] = "TTL_ERK"
    stexptim["stim_channel_device_name"] = "Spectra"
    stexptim["stim_channel_power_property_name"] = "Cyan_Level"


## Define the Tools that you are using for the experiment
from rtm_pymmcore.segmentation.stardist import SegmentatorStardist
from rtm_pymmcore.stimulation.base_stimulation import StimWholeFOV
from rtm_pymmcore.tracking.trackpy import TrackerTrackpy
from rtm_pymmcore.feature_extraction.erk_ktr import FE_ErkKtr
from rtm_pymmcore.feature_extraction.fe_for_optocheck import OptoCheckFE

segmentators = [
    {
        "name": "labels",
        "class": SegmentatorStardist(),
        "use_channel": 0,
        "save_tracked": True,
    },
]
stimulator = StimWholeFOV()
feature_extractor = FE_ErkKtr("labels")
tracker = TrackerTrackpy()
optocheck_fe = OptoCheckFE("labels")

from rtm_pymmcore.img_processing_pip import ImageProcessingPipeline

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=feature_extractor,
    tracker=tracker,
    stimulator=stimulator,
    feature_extractor_optocheck=optocheck_fe,
)

Pattern Name:  ramp1
10 at 1
20 at 2
30 at 3

Pattern Name:  ramp2
100 at 4
150 at 5
200 at 6
250 at 7

Found model '2D_versatile_fluo' for 'StarDist2D'.
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.479071, nms_thresh=0.3.
Directory C:\Users\Alex\Ausbildung\PhD_temp\test_exp\exp_test\raw already exists
Directory C:\Users\Alex\Ausbildung\PhD_temp\test_exp\exp_test\tracks already exists
Directory C:\Users\Alex\Ausbildung\PhD_temp\test_exp\exp_test\stim_mask already exists
Directory C:\Users\Alex\Ausbildung\PhD_temp\test_exp\exp_test\stim already exists
Directory C:\Users\Alex\Ausbildung\PhD_temp\test_exp\exp_test\particles already exists
Directory C:\Users\Alex\Ausbildung\PhD_temp\test_exp\exp_test\labels_ring already exists
Directory C:\Users\Alex\Ausbildung\PhD_temp\test_exp\exp_test\labels already exists
Directory C:\Users\Alex\Ausbildung\PhD_temp\test_exp\exp_test\optocheck already exists


### Load device and set startup channel

In [ ]:
mmc.loadSystemConfiguration(
    "E:\\pertzlab_mic_configs\\micromanager\\\Jungfrau\\TiFluoroJungfrau_w_TTL_DIGITALIO.cfg"
)
mmc.setConfig(groupName="System", configName="Startup")
mmc.setChannelGroup(channelGroup="TTL_ERK")

### GUI - Napari Micromanager

#### Load GUI

In [ ]:
### Base GUI ###
from napari_micromanager import MainWindow
import napari

viewer = napari.Viewer()
mm_wdg = MainWindow(viewer)
viewer.window.add_dock_widget(mm_wdg)
data_mda_fovs = None

### Add MDA widget for FOV selection ###
from pymmcore_widgets.mda import MDAWidget

mdawidget = MDAWidget(mmcore=mmc)
viewer.window.add_dock_widget(mdawidget)
load_from_file = False

#### Functions to break and re-connect link with GUI if manually broken

The following functions can be used to manually interrupt to connection between the GUI and the running rtm-pymmcore script. However, normally you don't need to execute them. 

In [ ]:
### Break connection
# mm_wdg._core_link.cleanup()

In [ ]:
### Manually reconnect pymmcore with napari-micromanager
from napari_micromanager._core_link import CoreViewerLink

mm_wdg._core_link = CoreViewerLink(viewer, mmc)

### Map Experiment to FOVs

#### If FOVs already saved - Reload them from file

In [73]:
import json

file = os.path.join(path, "fovs.json")
with open(file, "r") as f:
    data_mda_fovs = json.load(f)
data_mda_fovs = data_mda_fovs
load_from_file = True

### Use FOVs to generate dataframe for acquisition

In [77]:
n_fovs_simultaneously = TIME_BETWEEN_TIMESTEPS // TIME_PER_FOV
timesteps = range(N_FRAMES)


start_time = 0
if not load_from_file:
    data_mda_fovs = viewer.window._dock_widgets["MDA"].widget().value().stage_positions
    data_mda_fovs_dict = []
    for data_mda in data_mda_fovs:
        data_mda_fovs_dict.append(data_mda.model_dump())
    data_mda_fovs = data_mda_fovs_dict
    if data_mda_fovs is None:
        assert False, "No fovs selected. Please select fovs in the MDA widget"

if "channel_optocheck" not in locals():
    channel_optocheck = None
dfs = []
fovs = []
for fov_index, fov in enumerate(data_mda_fovs):
    fov_object = Fov(fov_index)
    fovs.append(fov_object)
    fov_group = fov_index // n_fovs_simultaneously
    start_time = fov_group * TIME_BETWEEN_TIMESTEPS * len(timesteps)
    if len(condition) == 1:
        condition_fov = condition[0]
    else:
        if fov_index // n_fovs_per_condition >= len(condition):
            print(
                f"Warning: Not enough conditions defined for the number of FOVs. Using the last condition {condition[-1]} for FOV {fov_index}."
            )
            condition_fov = condition[-1]
        # condition_fov = condition[fov_index // n_fovs_per_condition]
    for timestep in timesteps:
        row = {
            "fov_object": fov_object,
            "fov": fov_index,
            "fov_x": fov.get("x"),
            "fov_y": fov.get("y"),
            "fov_z": fov.get("z") if not USE_ONLY_PFS else None,
            "fov_name": str(fov_index) if fov["name"] is None else fov["name"],
            "timestep": timestep,
            "time": start_time + timestep * TIME_BETWEEN_TIMESTEPS,
            "cell_line": condition_fov,
            "channels": tuple(dataclasses.asdict(channel) for channel in channels),
            "fname": f"{str(fov_index).zfill(3)}_{str(timestep).zfill(5)}",
        }
        if channel_optocheck is not None:
            row["optocheck"] = True if timestep == N_FRAMES - 1 else False

            if isinstance(channel_optocheck, list):
                row["optocheck_channels"] = tuple(
                    dataclasses.asdict(channel) for channel in channel_optocheck
                )
            else:
                row["optocheck_channels"] = tuple(
                    [dataclasses.asdict(channel_optocheck)]
                )

        dfs.append(row)

df_acquire = pd.DataFrame(dfs)

print(f"Total Experiment Time: {df_acquire['time'].max()/3600}h")


n_fovs = len(data_mda_fovs)
n_stim_treatments = len(stim_exposures_timesteps)
if n_stim_treatments > 0:
    n_fovs_per_stim_condition = n_fovs // n_stim_treatments // len(np.unique(condition))
    stim_treatment_tot = []
    random.shuffle(stim_exposures_timesteps)
    if n_fovs_per_well is not None:
        for stim_treat in stim_exposures_timesteps:
            stim_treatment_tot.extend([stim_treat] * n_fovs_per_well)

    else:
        for fov_index in range(0, n_fovs_per_stim_condition):
            stim_treatment_tot.extend(stim_exposures_timesteps)
        random.shuffle(stim_treatment_tot)

        if n_fovs % n_stim_treatments != 0:
            print(
                f"Warning: Not equal number of fovs per stim condition. {n_fovs % n_stim_treatments} fovs will have repeated treatment"
            )
            stim_treatment_tot.extend(
                stim_exposures_timesteps[: n_fovs % n_stim_treatments]
            )
    print(f"Doing {n_fovs_per_stim_condition} experiment per stim condition")

    if len(condition) == 1:
        n_fovs_per_condition = n_fovs
    else:
        stim_treatment_tot = stim_treatment_tot * len(np.unique(condition))

    df_acquire = pd.merge(
        df_acquire, pd.DataFrame(stim_treatment_tot), left_on="fov", right_index=True
    )

    df_acquire["stim_exposure"] = 0

    for fov in df_acquire["fov"].unique():
        fov_data = df_acquire[df_acquire["fov"] == fov]

        stim_pattern = fov_data.iloc[0]

        if isinstance(stim_pattern["stim_timestep"], tuple) and isinstance(
            stim_pattern["stim_exposure_list"], tuple
        ):
            exposure_map = dict(
                zip(stim_pattern["stim_timestep"], stim_pattern["stim_exposure_list"])
            )

            for timestep in fov_data["timestep"]:
                if timestep in exposure_map:
                    mask = (df_acquire["fov"] == fov) & (
                        df_acquire["timestep"] == timestep
                    )
                    df_acquire.loc[mask, "stim_exposure"] = exposure_map[timestep]

    df_acquire["stim"] = df_acquire.apply(
        lambda row: (
            row["timestep"] in row["stim_timestep"] and row["stim_exposure"] > 0
        ),
        axis=1,
    )

df_acquire = df_acquire.dropna(axis=1, how="all")
pd.set_option("display.max_columns", None)
pd.set_option("display.expand_frame_repr", True)
df_acquire = df_acquire.sort_values(by=["time", "fov"]).reset_index(drop=True)
df_acquire[
    [
        "fov",
        "timestep",
        "time",
        "cell_line",
        "stim_power",
        "stim_exposure",
        "stim_exposure_list",
        "stim_timestep",
        "stim",
    ]
]

Total Experiment Time: 0.04388888888888889h
Doing 2 experiment per stim condition


,fov,timestep,time,cell_line,stim_power,stim_exposure,stim_exposure_list,stim_timestep,stim
0,0,0,0,2,11,0,"(100, 150, 200, 250, 300, 350, 400, 450)","(4, 5, 6, 7)",False
1,1,0,0,2,11,0,"(100, 150, 200, 250, 300, 350, 400, 450)","(4, 5, 6, 7)",False
2,0,1,2,2,11,0,"(100, 150, 200, 250, 300, 350, 400, 450)","(4, 5, 6, 7)",False
3,1,1,2,2,11,0,"(100, 150, 200, 250, 300, 350, 400, 450)","(4, 5, 6, 7)",False
4,0,2,4,2,11,0,"(100, 150, 200, 250, 300, 350, 400, 450)","(4, 5, 6, 7)",False
...,...,...,...,...,...,...,...,...,...
155,7,17,154,2,11,0,"(10, 20, 30)","(1, 2, 3)",False
156,6,18,156,2,11,0,"(10, 20, 30)","(1, 2, 3)",False
157,7,18,156,2,11,0,"(10, 20, 30)","(1, 2, 3)",False
158,6,19,158,2,11,0,"(10, 20, 30)","(1, 2, 3)",False


### Run experiment

In [ ]:
pymmcore_plus.configure_logging(stderr_level="WARNING")

for _ in range(0, SLEEP_BEFORE_EXPERIMENT_START_in_H * 3600):

    time.sleep(1)

from rtm_pymmcore.controller import Controller, Analyzer
from rtm_pymmcore.dmd import DMD
from queue import Queue

try:

    mm_wdg._core_link.cleanup()

except:
    pass


analyzer = Analyzer(pipeline)
queue = Queue()

controller = Controller(
    analyzer,
    mmc,
    queue,
)

controller.run(df_acquire)

print("Experiment finished")
time.sleep(60)


fovs_i_list = os.listdir(os.path.join(path, "tracks"))
fovs_i_list.sort()
dfs = []

for fov_i in fovs_i_list:

    track_file = os.path.join(path, "tracks", fov_i)
    df = pd.read_parquet(track_file)
    dfs.append(df)

pd.concat(dfs).to_parquet(os.path.join(path, "exp_data.parquet"))

Directory C:\Users\Alex\Ausbildung\PhD_temp\test_exp\exp_test\raw already exists
Directory C:\Users\Alex\Ausbildung\PhD_temp\test_exp\exp_test\tracks already exists
Directory C:\Users\Alex\Ausbildung\PhD_temp\test_exp\exp_test\stim_mask already exists
Directory C:\Users\Alex\Ausbildung\PhD_temp\test_exp\exp_test\stim already exists
Directory C:\Users\Alex\Ausbildung\PhD_temp\test_exp\exp_test\particles already exists
Directory C:\Users\Alex\Ausbildung\PhD_temp\test_exp\exp_test\labels_ring already exists
Directory C:\Users\Alex\Ausbildung\PhD_temp\test_exp\exp_test\labels already exists
Directory C:\Users\Alex\Ausbildung\PhD_temp\test_exp\exp_test\optocheck already exists


functional.py (238): The structure of `inputs` doesn't match the expected structure.
Expected: ['input']
Received: inputs=Tensor(shape=(1, 1024, 1024, 1))


Experiment finished
